In [94]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

In [95]:
data = pd.read_csv("../data/clean/build_dataset.csv")

In [96]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   gender               7043 non-null   str    
 1   senior_citizen       7043 non-null   str    
 2   partner              7043 non-null   str    
 3   dependents           7043 non-null   str    
 4   tenure_months        7043 non-null   int64  
 5   phone_service        7043 non-null   str    
 6   multiple_lines       7043 non-null   str    
 7   internet_service     7043 non-null   str    
 8   online_security      7043 non-null   str    
 9   online_backup        7043 non-null   str    
 10  device_protection    7043 non-null   str    
 11  tech_support         7043 non-null   str    
 12  streaming_tv         7043 non-null   str    
 13  streaming_movies     7043 non-null   str    
 14  contract             7043 non-null   str    
 15  paperless_billing    7043 non-null   str    
 16 

In [97]:
data.columns

Index(['gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months',
       'phone_service', 'multiple_lines', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing',
       'payment_method', 'monthly_charges', 'total_charges', 'churn_value',
       'cltv', 'tenure_group', 'avg_monthly_charge', 'charge_vs_avg',
       'payment_ratio', 'total_services', 'cost_per_service', 'high_risk',
       'engagement_score', 'tenure_x_contract', 'charge_x_risk',
       'log_total_charges', 'log_monthly_charges', 'log_cltv'],
      dtype='str')

In [98]:
data_model = data.copy()

In [99]:
X = data_model.drop("churn_value", axis=1)
y = data_model["churn_value"]

In [100]:
# stratify -> The churn rate (26%) is maintained in both sets; if stratify is not used, it may randomly split unevenly.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [101]:
print(f"Train : {X_train.shape[0]} rows | Test : {X_test.shape[0]} rows")
print(f"Train churn rate : {y_train.mean():.2%}")
print(f"Test  churn rate : {y_test.mean():.2%}\n")

Train : 5634 rows | Test : 1409 rows
Train churn rate : 26.54%
Test  churn rate : 26.54%



In [102]:
numeric_cols = [
    "tenure_months", "monthly_charges", "total_charges", "cltv", "avg_monthly_charge", "charge_vs_avg", "payment_ratio",
    "total_services", "cost_per_service", "high_risk", "engagement_score", "log_total_charges", "log_monthly_charges",
    "log_cltv", "tenure_x_contract", "charge_x_risk",
]

In [103]:
binary_cols = [
    "senior_citizen", "partner", "dependents", "phone_service", "paperless_billing", "online_security", "online_backup",
    "device_protection", "tech_support", "streaming_tv", "streaming_movies",
]

In [104]:
gender_cols = ["gender"]

In [105]:
ordinal_cols        = ["contract", "tenure_group"]
contract_categories = [["Month-to-month", "One year", "Two year"]]
tenure_categories   = [["New", "Growing", "Mature", "Loyal"]]

In [106]:
nominal_cols = ["multiple_lines", "internet_service", "payment_method"]

In [107]:
preprocessor = ColumnTransformer(transformers=[
    ("num",     StandardScaler(),
                numeric_cols),

    ("binary",  OrdinalEncoder(categories=[["No", "Yes"]] * len(binary_cols)),
                binary_cols),

    ("gender",  OrdinalEncoder(categories=[["Female", "Male"]]),
                gender_cols),

    ("ordinal", OrdinalEncoder(categories=contract_categories + tenure_categories),
                ordinal_cols),

    ("nominal", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
                nominal_cols),
],
remainder="drop"
)

In [108]:
X_train_encoded = preprocessor.fit_transform(X_train, y_train)
X_test_encoded = preprocessor.transform(X_test)

In [111]:
X_train = pd.DataFrame(X_train_encoded, columns=preprocessor.get_feature_names_out(), index=X_train.index)
X_test = pd.DataFrame(X_test_encoded, columns=preprocessor.get_feature_names_out(), index=X_test.index)

In [116]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 5634 entries, 4626 to 6017
Data columns (total 36 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   num__tenure_months                               5634 non-null   float64
 1   num__monthly_charges                             5634 non-null   float64
 2   num__total_charges                               5634 non-null   float64
 3   num__cltv                                        5634 non-null   float64
 4   num__avg_monthly_charge                          5634 non-null   float64
 5   num__charge_vs_avg                               5634 non-null   float64
 6   num__payment_ratio                               5634 non-null   float64
 7   num__total_services                              5634 non-null   float64
 8   num__cost_per_service                            5634 non-null   float64
 9   num__high_risk                             